# Gepard — tts inference demo

Runs the DPO checkpoint **`nineninesix/gepard-1.0`**.

The checkpoint is **self-describing**: it carries a `gepard_config.json`, so the
runner rebuilds the exact architecture (backbone, 32 audio heads, voice-cloning
compressor, `partial_rotary_factor`) from the checkpoint alone — no training
configs needed. Both repos are private, so authenticate first.

One runner — `GepardRunner`:
- default (`cfg_scale=1.0`) — plain single-pass generation
- `cfg_scale=2..3` — classifier-free guidance (better prosody/voice adherence, 2x compute on guided frames)

## Setup (Colab)

Skip this section if you run inside the `gepard-train` repo with `venv_infer`.

In [8]:
# HF auth — the model repo is private
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Clone the code and install the lean inference stack.
# NeMo goes FIRST, gepard[inference] LAST — it re-pins transformers==5.3.0
# on top of whatever NeMo downgraded (same order as scripts/setup.sh).
!git clone https://github.com/nineninesix-ai/gepard-inference.git
%cd gepard-inference
!pip install -q "nemo-toolkit[tts]==2.4.0"
!pip install -q -e .[inference]

## Imports

In [ ]:
from IPython.display import Audio as aplay
from gepard_inference import GepardRunner, Player

## Load the model

One call — everything (architecture, special tokens, codec geometry, text
repetition policy) comes from the checkpoint's `gepard_config.json`. Expect a
clean load: no missing / unexpected keys.

In [ ]:
CHECKPOINT = "nineninesix/gepard-1.0"

model = GepardRunner.from_checkpoint(CHECKPOINT)
player = Player.from_checkpoint(CHECKPOINT) 

## Reference voice

Encode a reference clip into FSQ codes — the Q-Former compressor turns it into
speaker prefix tokens. Bundled demo voices in `./ref_audio/`:
`audio_en.wav`, `audio_ru.wav`, `linda.wav`, `nurisa_en.wav`, `ulan_emo.wav`.

In [3]:
ref_codes = player.encode_reference("ref_audio/audio_ru.wav")
ref_codes.shape

torch.Size([1, 960, 32])

## Generate — plain runner

In [4]:
text = "Yesterday I decided I was going to be a responsible, organized person. I even said it out loud, which made it feel official."

tokens = model.generate(text, ref_codes=ref_codes, temperature=0.3)
# without a reference voice: tokens = model.generate(text, temperature=0.3)

In [5]:
sr, wave = player.decode(tokens)
aplay(wave, rate=sr)

## Generate — classifier-free guidance

`cfg_scale=3` sharpens text/voice adherence; `cfg_frames=N` limits guidance to
the first N frames (`None` = the whole utterance). Short phrases are where CFG
helps the most.

In [6]:
text = "Hi there! Great to finally meet you."

tokens = model.generate(text, ref_codes=ref_codes, temperature=0.3,
                            cfg_scale=3, cfg_frames=None)

In [7]:
sr, wave = player.decode(tokens)
aplay(wave, rate=sr)